# 03 — Partición estratificada de grabaciones y exportación del conjunto de prueba

## Descripción

Este notebook utiliza el **resumen de scores por grabación** generado por el notebook 02 para dividir las grabaciones completas en conjuntos de entrenamiento y prueba.

El procedimiento es el siguiente:

1. Se comprueba que el resumen y la base Hoplite contengan las mismas grabaciones.
2. Todas las grabaciones se ordenan según un score preliminar definido por el usuario.
3. La distribución completa se divide en cinco categorías de score o quintiles.
4. Se selecciona aleatoriamente una cantidad aproximadamente igual de grabaciones de cada quintil hasta completar `N_TEST_FILES`.
5. Todas las grabaciones restantes se asignan a entrenamiento.
6. Todas las ventanas de una grabación reciben el mismo split, evitando fuga de información entre train y test.
7. Se guardan las tablas de partición y se copian los audios de test para su anotación manual en Raven Pro.

La selección se realiza **a nivel de grabación completa**, no a nivel de ventanas.

## 1. Imports

In [ ]:
import json
import shutil
import sqlite3

from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

## 2. Configuración

In [ ]:
# ============================================================
# RUTAS
# ============================================================

# Resumen por grabación generado por el notebook 02.
# Ajustar el nombre si el archivo producido tiene otro nombre.
PATH_TO_RECORDING_SUMMARY = Path(r"/mnt/d/Taboga/Train/perch_embed/agile_classifier_inicial_pred_inicial_recording_summary.csv")

# Archivo SQLite de la base Hoplite completa.
PATH_TO_EMBEDDINGS = Path(
    r"/mnt/d/Taboga/Train/perch_embed/hoplite.sqlite"
)

# Carpeta donde se guardarán las tablas de la partición.
OUTPUT_DIR = Path(
    r"/mnt/d/Taboga/data_splits"
)

# Carpeta raíz donde están almacenados los audios originales.
SOURCE_AUDIO_DIR = Path(
    r"/mnt/d/Taboga/Train"
)

# Carpeta donde se copiarán los audios asignados a test.
TEST_AUDIO_DIR = Path(
    r"/mnt/d/Taboga/test_audio"
)


# ============================================================
# CONFIGURACIÓN DE LA PARTICIÓN
# ============================================================

# Semilla para reproducir exactamente la selección.
SEED = 42

# Número total de grabaciones que se asignarán a test.
N_TEST_FILES = 200

# Número de categorías o estratos de score.
N_SCORE_STRATA = 5

# Score por grabación utilizado para formar los quintiles.
# Recomendación: promedio de las mejores K ventanas.
STRATIFICATION_SCORE = "preliminary_top_k_mean_score"

# Nombres ordenados desde los scores menores hasta los mayores.
STRATUM_LABELS = [
    "Q1_very_low",
    "Q2_low",
    "Q3_medium",
    "Q4_high",
    "Q5_very_high",
]

# Si es True, elimina TEST_AUDIO_DIR antes de copiar los archivos.
# Usar con precaución: borra todo el contenido de esa carpeta.
CLEAR_TEST_AUDIO_DIR = False

In [ ]:
# ============================================================
# VERIFICACIONES DE CONFIGURACIÓN
# ============================================================

if N_SCORE_STRATA != len(STRATUM_LABELS):
    raise ValueError(
        "La cantidad de etiquetas en STRATUM_LABELS no coincide "
        "con N_SCORE_STRATA."
    )

if N_TEST_FILES < N_SCORE_STRATA:
    raise ValueError(
        "N_TEST_FILES debe ser igual o mayor que N_SCORE_STRATA."
    )

if len(set(STRATUM_LABELS)) != len(STRATUM_LABELS):
    raise ValueError("STRATUM_LABELS contiene nombres duplicados.")

if not PATH_TO_RECORDING_SUMMARY.exists():
    raise FileNotFoundError(
        "No se encontró el resumen por grabación:\n"
        f"{PATH_TO_RECORDING_SUMMARY}"
    )

if not PATH_TO_EMBEDDINGS.exists():
    raise FileNotFoundError(
        "No se encontró la base SQLite:\n"
        f"{PATH_TO_EMBEDDINGS}"
    )

print("Resumen por grabación:", PATH_TO_RECORDING_SUMMARY)
print("Base SQLite:", PATH_TO_EMBEDDINGS)
print("Número solicitado para test:", N_TEST_FILES)
print("Número de estratos:", N_SCORE_STRATA)
print("Score de estratificación:", STRATIFICATION_SCORE)

## 3. Cargar y validar el resumen por grabación del notebook 02

In [ ]:
recording_score_summary = pd.read_csv(
    PATH_TO_RECORDING_SUMMARY
)

required_summary_columns = {
    "recording_id",
    "filename",
    "n_windows",
    "preliminary_max_score",
    "preliminary_top_k_mean_score",
    "preliminary_p95_score",
    "preliminary_mean_score",
}

missing_columns = (
    required_summary_columns
    - set(recording_score_summary.columns)
)

if missing_columns:
    raise ValueError(
        "Faltan columnas requeridas en el resumen por grabación: "
        f"{sorted(missing_columns)}"
    )

if STRATIFICATION_SCORE not in recording_score_summary.columns:
    raise ValueError(
        f"La columna configurada como STRATIFICATION_SCORE "
        f"no existe: {STRATIFICATION_SCORE}"
    )

recording_score_summary["recording_id"] = (
    pd.to_numeric(
        recording_score_summary["recording_id"],
        errors="raise",
    ).astype(int)
)

if recording_score_summary["recording_id"].duplicated().any():
    duplicated = recording_score_summary.loc[
        recording_score_summary["recording_id"].duplicated(
            keep=False
        )
    ]
    raise ValueError(
        "El resumen contiene recording_id duplicados:\n"
        f"{duplicated.head(20)}"
    )

if recording_score_summary[STRATIFICATION_SCORE].isna().any():
    raise ValueError(
        f"La columna {STRATIFICATION_SCORE} contiene valores faltantes."
    )

if not np.isfinite(
    recording_score_summary[STRATIFICATION_SCORE].to_numpy(
        dtype=float
    )
).all():
    raise ValueError(
        f"La columna {STRATIFICATION_SCORE} contiene valores no finitos."
    )

print(
    "Grabaciones en el resumen:",
    f"{len(recording_score_summary):,}",
)

display(recording_score_summary.head())

## 4. Cargar la metadata de la base Hoplite

In [ ]:
with sqlite3.connect(PATH_TO_EMBEDDINGS) as conn:
    tables = pd.read_sql_query(
        """
        SELECT name
        FROM sqlite_master
        WHERE type = 'table'
        ORDER BY name
        """,
        conn,
    )

    window_metadata = pd.read_sql_query(
        """
        SELECT
            w.id AS window_id,
            w.recording_id,
            w.offsets,
            r.filename,
            r.deployment_id
        FROM windows AS w
        INNER JOIN recordings AS r
            ON w.recording_id = r.id
        """,
        conn,
    )

print("Tablas contenidas en la base Hoplite:")
display(tables)

print(
    "Ventanas registradas:",
    f"{len(window_metadata):,}",
)

In [ ]:
# Tabla única de grabaciones registrada en Hoplite.
all_recordings = (
    window_metadata[
        [
            "recording_id",
            "filename",
            "deployment_id",
        ]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

all_recordings["recording_id"] = (
    all_recordings["recording_id"].astype(int)
)

if all_recordings["recording_id"].duplicated().any():
    duplicated = all_recordings.loc[
        all_recordings["recording_id"].duplicated(keep=False)
    ]
    raise ValueError(
        "Un recording_id aparece asociado con más de una grabación:\n"
        f"{duplicated.head(20)}"
    )

if window_metadata["window_id"].duplicated().any():
    raise ValueError("La tabla windows contiene window_id duplicados.")

print(
    "Total de grabaciones en SQLite:",
    f"{len(all_recordings):,}",
)

## 5. Verificar correspondencia entre el resumen y SQLite

In [ ]:
database_ids = set(all_recordings["recording_id"])
summary_ids = set(recording_score_summary["recording_id"])

missing_in_summary = database_ids - summary_ids
unexpected_in_summary = summary_ids - database_ids

if missing_in_summary:
    raise ValueError(
        f"{len(missing_in_summary)} grabaciones de SQLite no aparecen "
        "en el resumen por grabación. Primeros IDs: "
        f"{sorted(missing_in_summary)[:20]}"
    )

if unexpected_in_summary:
    raise ValueError(
        f"{len(unexpected_in_summary)} grabaciones del resumen no aparecen "
        "en SQLite. Primeros IDs: "
        f"{sorted(unexpected_in_summary)[:20]}"
    )

if len(recording_score_summary) != len(all_recordings):
    raise ValueError(
        "El número de grabaciones del resumen no coincide con el número "
        "registrado en SQLite."
    )

print(
    "El resumen y SQLite contienen exactamente los mismos recording_id."
)

In [ ]:
# Comprobar que cada recording_id está asociado al mismo nombre de archivo
# en el resumen y en la base Hoplite.
filename_check = all_recordings.merge(
    recording_score_summary[
        [
            "recording_id",
            "filename",
        ]
    ],
    on="recording_id",
    how="inner",
    suffixes=("_sqlite", "_summary"),
    validate="one_to_one",
)

filename_check["filename_sqlite_normalized"] = (
    filename_check["filename_sqlite"]
    .astype(str)
    .str.replace("\\", "/", regex=False)
)

filename_check["filename_summary_normalized"] = (
    filename_check["filename_summary"]
    .astype(str)
    .str.replace("\\", "/", regex=False)
)

filename_mismatch = filename_check[
    filename_check["filename_sqlite_normalized"]
    != filename_check["filename_summary_normalized"]
]

if not filename_mismatch.empty:
    raise ValueError(
        "Algunos recording_id están asociados con nombres diferentes "
        "en SQLite y en el resumen:\n"
        f"{filename_mismatch.head(20)}"
    )

print("Todos los nombres de archivo coinciden.")

## 6. Construir la tabla maestra de grabaciones

In [ ]:
# SQLite se utiliza como fuente oficial para filename y deployment_id.
summary_without_repeated_metadata = (
    recording_score_summary.drop(
        columns=[
            "filename",
            "deployment_id",
        ],
        errors="ignore",
    )
)

recording_pool = all_recordings.merge(
    summary_without_repeated_metadata,
    on="recording_id",
    how="left",
    validate="one_to_one",
)

if recording_pool[STRATIFICATION_SCORE].isna().any():
    missing = recording_pool.loc[
        recording_pool[STRATIFICATION_SCORE].isna(),
        [
            "recording_id",
            "filename",
        ],
    ]
    raise ValueError(
        "Hay grabaciones sin score de estratificación:\n"
        f"{missing.head(20)}"
    )

if N_TEST_FILES > len(recording_pool):
    raise ValueError(
        "N_TEST_FILES es mayor que el número total de grabaciones "
        "disponibles."
    )

print(
    "Grabaciones disponibles para la partición:",
    f"{len(recording_pool):,}",
)

display(
    recording_pool[
        [
            "recording_id",
            "filename",
            "n_windows",
            "preliminary_max_score",
            "preliminary_top_k_mean_score",
            "preliminary_p95_score",
        ]
    ].head()
)

## 7. Crear los cinco estratos de score

Se utiliza un ranking antes de aplicar `qcut`. Esto garantiza la formación de cinco grupos de tamaños aproximadamente iguales incluso cuando varias grabaciones tienen exactamente el mismo score.

La mezcla previa del orden únicamente determina cómo se resuelven los empates; los scores originales no se modifican.

In [ ]:
# Mezclar temporalmente el orden permite resolver los empates en score
# de forma aleatoria, pero reproducible.
score_ranking = (recording_pool[["recording_id",STRATIFICATION_SCORE,]].sample(frac=1, random_state=SEED,).copy())

# Los scores menores reciben rangos menores.
score_ranking["_score_rank"] = (
    score_ranking[STRATIFICATION_SCORE].rank(
        method="first",
        ascending=True,
    )
)

score_ranking["score_stratum"] = pd.qcut(
    score_ranking["_score_rank"],
    q=N_SCORE_STRATA,
    labels=STRATUM_LABELS,
)

recording_pool = recording_pool.merge(
    score_ranking[
        [
            "recording_id",
            "score_stratum",
        ]
    ],
    on="recording_id",
    how="left",
    validate="one_to_one",
)

if recording_pool["score_stratum"].isna().any():
    raise ValueError(
        "No fue posible asignar un estrato de score a todas las "
        "grabaciones."
    )

created_strata = recording_pool["score_stratum"].nunique()

if created_strata != N_SCORE_STRATA:
    raise ValueError(
        f"Se esperaban {N_SCORE_STRATA} estratos, pero se crearon "
        f"{created_strata}."
    )

print("Cantidad de grabaciones por estrato:")

display(
    recording_pool["score_stratum"]
    .value_counts(sort=False)
    .rename("n_recordings")
    .reset_index()
)

In [ ]:
score_strata_summary = (
    recording_pool
    .groupby(
        "score_stratum",
        observed=True,
    )
    .agg(
        n_recordings=("recording_id", "size"),
        minimum_score=(STRATIFICATION_SCORE, "min"),
        median_score=(STRATIFICATION_SCORE, "median"),
        mean_score=(STRATIFICATION_SCORE, "mean"),
        maximum_score=(STRATIFICATION_SCORE, "max"),
    )
    .reset_index()
)

display(score_strata_summary)

## 8. Distribuir `N_TEST_FILES` entre los estratos

In [ ]:
base_n, remainder = divmod(
    N_TEST_FILES,
    N_SCORE_STRATA,
)

test_n_by_stratum = {
    stratum: base_n
    for stratum in STRATUM_LABELS
}

# Cuando N_TEST_FILES no es divisible exactamente entre los estratos,
# las unidades restantes se asignan aleatoriamente a algunos estratos.
allocation_rng = np.random.default_rng(SEED)

if remainder > 0:
    strata_receiving_extra = allocation_rng.choice(
        STRATUM_LABELS,
        size=remainder,
        replace=False,
    )

    for stratum in strata_receiving_extra:
        test_n_by_stratum[str(stratum)] += 1

population_n_by_stratum = (
    recording_pool["score_stratum"]
    .value_counts(sort=False)
    .reindex(STRATUM_LABELS)
    .astype(int)
    .to_dict()
)

allocation_rows = []

for stratum in STRATUM_LABELS:
    population_n = population_n_by_stratum[stratum]
    requested_n = test_n_by_stratum[stratum]

    if requested_n > population_n:
        raise ValueError(
            f"El estrato {stratum} contiene solamente {population_n} "
            f"grabaciones, pero se solicitaron {requested_n} para test."
        )

    allocation_rows.append(
        {
            "score_stratum": stratum,
            "population_n": population_n,
            "test_n_requested": requested_n,
            "train_n_expected": population_n - requested_n,
            "test_selection_fraction": requested_n / population_n,
        }
    )

test_allocation = pd.DataFrame(allocation_rows)

if test_allocation["test_n_requested"].sum() != N_TEST_FILES:
    raise RuntimeError(
        "La suma de la asignación por estrato no coincide con "
        "N_TEST_FILES."
    )

print("Asignación solicitada por estrato:")
display(test_allocation)

## 9. Seleccionar aleatoriamente las grabaciones de test

In [ ]:
selected_test_parts = []

for stratum_index, stratum in enumerate(STRATUM_LABELS):
    stratum_pool = recording_pool[
        recording_pool["score_stratum"] == stratum
    ].copy()

    n_to_select = test_n_by_stratum[stratum]

    selected_stratum = stratum_pool.sample(
        n=n_to_select,
        replace=False,
        random_state=SEED + 100 + stratum_index,
    ).copy()

    selected_test_parts.append(selected_stratum)

test_recordings = pd.concat(
    selected_test_parts,
    ignore_index=True,
)

test_recordings["split"] = "test"

# Se conserva esta columna para mantener compatibilidad con tablas previas.
test_recordings["test_component"] = "score_stratified"

test_recordings = (
    test_recordings
    .sort_values(
        [
            "score_stratum",
            STRATIFICATION_SCORE,
        ],
        ascending=[
            True,
            False,
        ],
    )
    .reset_index(drop=True)
)

print(
    "Total de grabaciones seleccionadas para test:",
    f"{len(test_recordings):,}",
)

In [ ]:
# Añadir metadata del diseño de muestreo.
population_n_map = test_allocation.set_index(
    "score_stratum"
)["population_n"].to_dict()

test_n_map = test_allocation.set_index(
    "score_stratum"
)["test_n_requested"].to_dict()

selection_fraction_map = test_allocation.set_index(
    "score_stratum"
)["test_selection_fraction"].to_dict()

recording_pool["stratum_population_n"] = (
    recording_pool["score_stratum"].map(population_n_map).astype(int)
)

recording_pool["stratum_test_n"] = (
    recording_pool["score_stratum"].map(test_n_map).astype(int)
)

recording_pool["test_selection_probability"] = (
    recording_pool["score_stratum"].map(selection_fraction_map).astype(float)
)

# Incorporar estas columnas al dataframe test ya seleccionado.
test_recordings = (
    test_recordings.drop(
        columns=[
            "stratum_population_n",
            "stratum_test_n",
            "test_selection_probability",
        ],
        errors="ignore",
    )
    .merge(
        recording_pool[
            [
                "recording_id",
                "stratum_population_n",
                "stratum_test_n",
                "test_selection_probability",
            ]
        ],
        on="recording_id",
        how="left",
        validate="one_to_one",
    )
)

## 10. Definir entrenamiento como todas las grabaciones restantes

In [ ]:
test_recording_ids = set(test_recordings["recording_id"])

train_recordings = recording_pool[
    ~recording_pool["recording_id"].isin(test_recording_ids)
].copy()

train_recordings["split"] = "train"
train_recordings["test_component"] = pd.NA

print("Grabaciones en train:", f"{len(train_recordings):,}")
print("Grabaciones en test:", f"{len(test_recordings):,}")

## 11. Verificaciones críticas de la partición

In [ ]:
train_ids = set(train_recordings["recording_id"])
test_ids = set(test_recordings["recording_id"])
all_ids = set(recording_pool["recording_id"])

if not train_ids.isdisjoint(test_ids):
    overlap = train_ids.intersection(test_ids)
    raise ValueError(
        "Hay grabaciones presentes simultáneamente en train y test: "
        f"{sorted(overlap)[:20]}"
    )

if train_ids | test_ids != all_ids:
    missing = all_ids - (train_ids | test_ids)
    raise ValueError(
        "Hay grabaciones que no fueron asignadas: "
        f"{sorted(missing)[:20]}"
    )

unexpected = (train_ids | test_ids) - all_ids
if unexpected:
    raise ValueError(
        "La partición contiene IDs que no están en la base: "
        f"{sorted(unexpected)[:20]}"
    )

if len(train_recordings) + len(test_recordings) != len(recording_pool):
    raise ValueError(
        "El número de grabaciones de train y test no suma el total "
        "de la base."
    )

if len(test_recordings) != N_TEST_FILES:
    raise ValueError(
        f"Se esperaban {N_TEST_FILES} grabaciones en test, pero se "
        f"obtuvieron {len(test_recordings)}."
    )

if train_recordings["recording_id"].duplicated().any():
    raise ValueError("Hay recording_id duplicados en train.")

if test_recordings["recording_id"].duplicated().any():
    raise ValueError("Hay recording_id duplicados en test.")

print("Todas las verificaciones generales fueron superadas.")

In [ ]:
actual_test_counts = (
    test_recordings["score_stratum"]
    .value_counts(sort=False)
    .reindex(STRATUM_LABELS)
    .astype(int)
)

expected_test_counts = pd.Series(
    test_n_by_stratum,
    name="expected_n",
).reindex(STRATUM_LABELS)

stratum_count_check = pd.DataFrame(
    {
        "expected_n": expected_test_counts,
        "actual_n": actual_test_counts,
    }
)

stratum_count_check["difference"] = (
    stratum_count_check["actual_n"]
    - stratum_count_check["expected_n"]
)

display(stratum_count_check)

if not (stratum_count_check["difference"] == 0).all():
    raise ValueError(
        "La cantidad seleccionada en uno o más estratos no coincide "
        "con la cantidad esperada."
    )

print(
    "La selección por estrato coincide exactamente con la asignación "
    "solicitada."
)

In [ ]:
partition_summary = (
    pd.concat(
        [
            train_recordings,
            test_recordings,
        ],
        ignore_index=True,
    )
    .groupby(
        [
            "split",
            "score_stratum",
        ],
        observed=True,
    )
    .agg(
        n_recordings=("recording_id", "size"),
        minimum_score=(STRATIFICATION_SCORE, "min"),
        median_score=(STRATIFICATION_SCORE, "median"),
        maximum_score=(STRATIFICATION_SCORE, "max"),
    )
    .reset_index()
)

print("Total de grabaciones:", f"{len(recording_pool):,}")
print("Train:", f"{len(train_recordings):,}")
print("Test:", f"{len(test_recordings):,}")

display(partition_summary)

## 12. Construir la tabla maestra `recording_split.csv`

In [ ]:
split_table = pd.concat(
    [
        train_recordings,
        test_recordings,
    ],
    ignore_index=True,
    sort=False,
)

split_table = (
    split_table
    .sort_values(
        [
            "split",
            "score_stratum",
            "recording_id",
        ]
    )
    .reset_index(drop=True)
)

if split_table["recording_id"].duplicated().any():
    raise ValueError(
        "La tabla maestra contiene recording_id duplicados."
    )

created_utc = datetime.now(timezone.utc).isoformat()

split_table["split_seed"] = SEED
split_table["n_test_files_requested"] = N_TEST_FILES
split_table["n_score_strata"] = N_SCORE_STRATA
split_table["stratification_score"] = STRATIFICATION_SCORE
split_table["stratification_method"] = (
    "score_rank_quantiles_random_within_stratum"
)
split_table["created_utc"] = created_utc

print(
    split_table.groupby(
        [
            "split",
            "score_stratum",
        ],
        observed=True,
        dropna=False,
    ).size()
)

## 13. Asignar todas las ventanas al split de su grabación

Una vez que una grabación se asigna a train o test, **todas sus ventanas** reciben la misma asignación.

In [ ]:
window_split = window_metadata.merge(
    split_table[
        [
            "recording_id",
            "split",
            "test_component",
            "score_stratum",
            STRATIFICATION_SCORE,
        ]
    ],
    on="recording_id",
    how="left",
    validate="many_to_one",
)

if window_split["split"].isna().any():
    missing = window_split.loc[
        window_split["split"].isna(),
        "recording_id",
    ].unique()
    raise ValueError(
        "Hay ventanas cuyos recording_id no fueron asignados: "
        f"{missing[:20]}"
    )

train_windows = window_split[
    window_split["split"] == "train"
].copy()

test_windows = window_split[
    window_split["split"] == "test"
].copy()

print("Ventanas en train:", f"{len(train_windows):,}")
print("Ventanas en test:", f"{len(test_windows):,}")

In [ ]:
train_window_ids = set(train_windows["window_id"])
test_window_ids = set(test_windows["window_id"])
all_window_ids = set(window_metadata["window_id"])

if not train_window_ids.isdisjoint(test_window_ids):
    raise ValueError(
        "Hay ventanas presentes simultáneamente en train y test."
    )

if train_window_ids | test_window_ids != all_window_ids:
    missing_windows = all_window_ids - (
        train_window_ids | test_window_ids
    )
    raise ValueError(
        "Hay ventanas que no fueron asignadas: "
        f"{sorted(missing_windows)[:20]}"
    )

if len(train_windows) + len(test_windows) != len(window_metadata):
    raise ValueError(
        "El número de ventanas de train y test no suma el total "
        "registrado en SQLite."
    )

print("La partición de ventanas fue verificada correctamente.")

## 14. Guardar las tablas y el manifiesto de reproducibilidad

In [ ]:
OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

# Tablas por grabación.
split_table.to_csv(
    OUTPUT_DIR / "recording_split.csv",
    index=False,
)

train_recordings.to_csv(
    OUTPUT_DIR / "train_recordings.csv",
    index=False,
)

test_recordings.to_csv(
    OUTPUT_DIR / "test_recordings.csv",
    index=False,
)

score_strata_summary.to_csv(
    OUTPUT_DIR / "score_strata_summary.csv",
    index=False,
)

test_allocation.to_csv(
    OUTPUT_DIR / "test_allocation_by_stratum.csv",
    index=False,
)

partition_summary.to_csv(
    OUTPUT_DIR / "partition_summary.csv",
    index=False,
)

# Tablas por ventana.
train_windows.to_csv(
    OUTPUT_DIR / "train_windows.csv",
    index=False,
)

test_windows.to_csv(
    OUTPUT_DIR / "test_windows.csv",
    index=False,
)

# Listas simplificadas de archivos.
train_recordings[
    [
        "recording_id",
        "filename",
        "score_stratum",
        STRATIFICATION_SCORE,
    ]
].to_csv(
    OUTPUT_DIR / "train_filenames.csv",
    index=False,
)

test_recordings[
    [
        "recording_id",
        "filename",
        "score_stratum",
        STRATIFICATION_SCORE,
    ]
].to_csv(
    OUTPUT_DIR / "test_filenames.csv",
    index=False,
)

print("Tablas guardadas en:", OUTPUT_DIR)

In [ ]:
split_manifest = {
    "created_utc": created_utc,
    "seed": SEED,
    "recording_summary_path": str(
        PATH_TO_RECORDING_SUMMARY.resolve()
    ),
    "sqlite_path": str(PATH_TO_EMBEDDINGS.resolve()),
    "n_total_recordings": int(len(recording_pool)),
    "n_train_recordings": int(len(train_recordings)),
    "n_test_recordings": int(len(test_recordings)),
    "n_score_strata": N_SCORE_STRATA,
    "stratum_labels": STRATUM_LABELS,
    "stratification_score": STRATIFICATION_SCORE,
    "stratification_method": (
        "rank-based quantiles with reproducible random tie resolution"
    ),
    "test_selection_method": (
        "random sampling without replacement within each score stratum"
    ),
    "test_n_by_stratum": {
        key: int(value)
        for key, value in test_n_by_stratum.items()
    },
}

manifest_path = OUTPUT_DIR / "split_manifest.json"

with open(
    manifest_path,
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        split_manifest,
        file,
        indent=2,
        ensure_ascii=False,
    )

print("Manifest guardado en:", manifest_path)

## 15. Exportar los audios del test para anotación manual en Raven Pro

El nombre exportado incluye `recording_id` para evitar sobrescribir archivos que tengan el mismo nombre base en carpetas diferentes.

In [ ]:
if CLEAR_TEST_AUDIO_DIR and TEST_AUDIO_DIR.exists():
    shutil.rmtree(TEST_AUDIO_DIR)

TEST_AUDIO_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

copy_log = []

for _, row in test_recordings.iterrows():
    recording_id = int(row["recording_id"])
    filename = str(row["filename"])
    score_stratum = str(row["score_stratum"])
    stratification_score_value = float(
        row[STRATIFICATION_SCORE]
    )

    source_file = Path(filename)
    if not source_file.is_absolute():
        source_file = SOURCE_AUDIO_DIR / source_file

    exported_filename = (
        f"{recording_id:06d}__{Path(filename).name}"
    )

    destination_file = TEST_AUDIO_DIR / exported_filename

    log_row = {
        "recording_id": recording_id,
        "original_filename": filename,
        "exported_filename": exported_filename,
        "score_stratum": score_stratum,
        STRATIFICATION_SCORE: stratification_score_value,
        "source": str(source_file),
        "destination": str(destination_file),
    }

    if not source_file.exists():
        log_row["status"] = "not_found"
        copy_log.append(log_row)
        continue

    if destination_file.exists():
        log_row["status"] = "already_exists"
        copy_log.append(log_row)
        continue

    shutil.copy2(
        source_file,
        destination_file,
    )

    log_row["status"] = "copied"
    copy_log.append(log_row)

copy_log = pd.DataFrame(copy_log)

print(copy_log["status"].value_counts(dropna=False))

copy_log.to_csv(
    TEST_AUDIO_DIR / "copy_log.csv",
    index=False,
)

missing_audio_count = int(
    copy_log["status"].eq("not_found").sum()
)

if missing_audio_count > 0:
    print(
        f"ADVERTENCIA: {missing_audio_count} audios de test no fueron "
        "encontrados. Revisar copy_log.csv."
    )
else:
    print("Todos los audios de test fueron localizados.")

## Archivos producidos

La carpeta `OUTPUT_DIR` contendrá:

- `recording_split.csv`: tabla maestra utilizada por el notebook 04.
- `train_recordings.csv` y `test_recordings.csv`.
- `train_windows.csv` y `test_windows.csv`.
- `train_filenames.csv` y `test_filenames.csv`.
- `score_strata_summary.csv`: rangos de scores de los quintiles.
- `test_allocation_by_stratum.csv`: cantidad solicitada y probabilidad de selección por quintil.
- `partition_summary.csv`: resumen de train y test por quintil.
- `split_manifest.json`: configuración completa de reproducibilidad.

La carpeta `TEST_AUDIO_DIR` contendrá los audios exportados y `copy_log.csv`.